# Day 2 Lab: Data Pipeline in Databricks

## Objective
- Build a data pipeline using PySpark + SQL
- Perform transformations and aggregations
- Understand execution plans


## Step 1: Create Sample Data

In [ ]:
orders_data = [
 (1,101,'2024-01-01',250,'completed'),
 (2,102,'2024-01-02',300,'completed'),
 (3,103,'2024-01-02',150,'cancelled'),
 (4,101,'2024-01-03',400,'completed'),
 (5,104,'2024-01-04',200,'completed'),
 (6,105,'2024-01-05',500,'returned')
]

customers_data = [
 (101,'Alice','Bangalore','2023-01-10'),
 (102,'Bob','Chennai','2023-02-15'),
 (103,'Charlie','Hyderabad','2023-03-20'),
 (104,'David','Bangalore','2023-04-25'),
 (105,'Eva','Delhi','2023-05-30')
]

orders_df = spark.createDataFrame(orders_data, ['order_id','customer_id','order_date','amount','status'])
customers_df = spark.createDataFrame(customers_data, ['customer_id','name','city','signup_date'])

display(orders_df)
display(customers_df)

## Step 2: Data Cleaning

In [ ]:
from pyspark.sql.functions import col

orders_clean = orders_df.filter(col('status') == 'completed') \
    .withColumn('amount', col('amount').cast('int'))

display(orders_clean)

## Step 3: Join Datasets

In [ ]:
joined_df = orders_clean.join(customers_df, on='customer_id', how='inner')

display(joined_df)

## Step 4: Aggregations

In [ ]:
from pyspark.sql.functions import sum, count

agg_df = joined_df.groupBy('customer_id','name','city') \
    .agg(
        sum('amount').alias('total_spent'),
        count('order_id').alias('total_orders')
    )

display(agg_df)

## Step 5: Spark SQL

In [ ]:
joined_df.createOrReplaceTempView('orders_customers')

In [ ]:
%sql
SELECT city, COUNT(*) as total_orders, SUM(amount) as revenue
FROM orders_customers
GROUP BY city
ORDER BY revenue DESC

## Step 6: Execution Plan

In [ ]:
agg_df.explain(True)

## Step 7: Write Output

In [ ]:
agg_df.write.mode('overwrite').parquet('/tmp/customer_spend')

## Bonus: Top Customers

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy(col('total_spent').desc())

top_customers = agg_df.withColumn('rank', row_number().over(window_spec)) \
    .filter(col('rank') <= 2)

display(top_customers)